>В сферах машинного обучения (ML) и искусственного интеллекта (AI) работа с вещественными числами — это абсолютная основа, так как нейросети и алгоритмы полностью построены на математике, матрицах и статистике.

Выбор точности (float или double) и понимание ошибок округления играют критическую роль для скорости обучения моделей и экономии миллионов долларов на серверах.

**Вещественные числа (float, FP32)** - хранятся в памяти компьютера в виде чисел с плавающей запятой по стандарту IEEE 754.

Компьютер переводит число в двоичную систему, записывает его в научном формате и делит выделенные биты (обычно 32 или 64 бита) на три четкие части.

1. Структура хранения float (на примере 32 бит)

Любое вещественное число раскладывается на три компонента:

- **Знак (Sign)** — 1 бит. Определяет, положительное число (`0`) или отрицательное (`1`).
- **Экспонента (Exponent)** — 8 бит. Определяет положение запятой (масштаб числа). К ней прибавляется смещение (127), чтобы избегать отрицательных степеней.
- **Мантисса (Mantissa / Fraction)** — 23 бита. Хранит значащие цифры числа.

Пример: как компьютер видит число 8.5

1. **Перевод в двоичную систему**: Число 8.5 превращается в 1000.1₂.
2. **Нормализация**: Запятая сдвигается так, чтобы слева осталась одна единица: 1.0001₂ × 2³.
3. **Запись в память**:
    - **Знак**: `0` (число положительное).
    - **Экспонента**: Степень 3 плюс смещение 127 равняется 130. В двоичном виде это `10000010`.
    - **Мантисса**: Единица перед запятой отбрасывается ради экономии места. В память идут только цифры после запятой (`0001`), а остаток забивается нулями: `00010000000000000000000`.

В итоге в памяти компьютера число 8.5 выглядит так: `0 10000010 00010000000000000000000`.

Почему возникают ошибки округления?

Многие десятичные дроби (например, 0.1 или 0.2) невозможно точно записать в двоичной системе. В двоичном виде 0.1 превращается в бесконечную периодическую дробь 0.0001100110011...₂.

Так как память компьютера ограничена 32 или 64 битами, бесконечный хвост просто **отсекается**. Из-за этого в программировании возникает известный парадокс, когда `0.1 + 0.2` не равняется точно `0.3`, а выдает `0.30000000000000004`.

---

2. Как компьютер понимает NaN и Infinity

Стандарт **IEEE 754** резервирует экстремальные значения экспоненты для особых ситуаций. Чтобы компьютер понял, что перед ним бесконечность или ошибка, **все биты экспоненты заполняются единицами (`11111111`)**.

Компьютер определяет статус числа по состоянию мантиссы:

Положительная бесконечность (`+Infinity`)

- Возникает при делении положительного числа на ноль (например, `1.0 / 0.0`).
- **Знак:** `0` (плюс)
- **Экспонента:** все единицы
- **Мантисса:** все нули

Отрицательная бесконечность (`-Infinity`)

- Возникает при делении отрицательного числа на ноль (например, `-1.0 / 0.0`).
- **Знак:** `1` (минус)
- **Экспонента:** все единицы
- **Мантисса:** все нули

NaN (Not a Number — не число)

- Возникает при математически неопределенных операциях, например: `0.0 / 0.0`, `Infinity - Infinity` или извлечение квадратного корня из отрицательного числа.
- **Знак:** может быть любым
- **Экспонента:** все единицы
- **Мантисса:** **хотя бы один бит не равен нулю** (любая ненулевая комбинация)

---

3. Как избежать ошибок с округлением дробей в реальном коде

Из-за двоичной природы `float` и `double` операции вроде `0.1 + 0.2` выдают `0.30000000000000004`. В финансовом, медицинском или инженерном софте это недопустимо.

Вот 4 способа борьбы с этой проблемой:

Способ 1. Использовать специализированные типы данных

Для точных десятичных вычислений (особенно в деньгах) используйте встроенные библиотеки, которые хранят числа в десятичном виде, а не в двоичном:

- **Python:** модуль `decimal` (класс `Decimal`)
- **Java / C#:** класс `BigDecimal` / тип `decimal`
- **JavaScript:** библиотека `big.js` или `bignumber.js`

In [ ]:
# Пример на Python
from decimal import Decimal

# Неправильно: 0.1 + 0.2 -> 0.30000000000000004
# Правильно:
result = Decimal('0.1') + Decimal('0.2')  # Важно передавать именно строки!
print(result)  # Выведет ровно 0.3


Способ 2. Сравнение с учетом погрешности (Epsilon)

Никогда не сравнивайте вещественные числа через точное равенство (`if (a == b)`). Вместо этого проверяйте, что разница между ними ничтожно мала. Эта минимальная разница называется **epsilon**.

In [ ]:
import math

a = 0.1 + 0.2
b = 0.3

# Неправильно: if a == b:
# Правильно:
if math.isclose(a, b):
    print("Числа равны с учетом погрешности")

##############################################
def are_close(a: float, b: float, rel_tol: float = 1e-09, abs_tol: float = 0.0) -> bool:
    """
    Безопасное сравнение двух вещественных чисел.
    
    :param a: Первое число
    :param b: Второе число
    :param rel_tol: Относительный допуск (максимально допустимая разница в процентах)
    :param abs_tol: Абсолютный допуск (минимальный порог для чисел близких к нулю)
    """
    # Считаем абсолютную разницу между числами
    diff = abs(a - b)
    
    # Проверяем, укладывается ли разница в рамки допусков
    return diff <= max(rel_tol * max(abs(a), abs(b)), abs_tol)

# --- Примеры использования ---

x = 0.1 + 0.2
y = 0.3

print(x == y)          # False (из-за ошибки округления 0.30000000000000004)
print(are_close(x, y)) # True (числа безопасно совпали)



Как это работает:

1. `abs(a - b)` находит физическое расстояние между числами на числовой прямой.
2. `rel_tol * max(abs(a), abs(b))` масштабирует допустимую ошибку под размер самих чисел (по умолчанию это 9 знаков после запятой).
3. `abs_tol` спасает, когда оба числа стремятся к нулю, где относительная погрешность перестает адекватно работать.

---

Способ 3. Перевод в целые числа (центы, копейки)

Если вы пишете интернет-магазин, храните и считайте все цены не в рублях/долларах, а в копейках/центах.

- Вместо `10.50` храните целое число `1050`.
- Все математические операции (сложение, скидки) делайте в целых числах.
- Делите на 100 только в самый последний момент — при выводе цены на экран пользователю.

Способ 4. Округление при выводе

Если микроскопическая погрешность внутри вычислений не влияет на логику программы, а смущает только пользователя на экране, просто форматируйте строку вывода:

python

```
value = 0.1 + 0.2
print(f"{value:.2f}")
```

---

Если точности вещественного числа не хватит, маленькие градиенты просто округлятся до нуля. Это называется проблемой затухания градиентов (vanishing gradients), и нейросеть перестанет учиться.

### Квантование и экономия памяти (Смена типов данных)

Раньше нейросети обучались на стандартных числах двойной точности (`double` / FP64) или одинарной точности (`float` / FP32). Но современные модели (вроде LLM — ChatGPT, Claude, Llama) весят терабайты, если использовать FP32.

Чтобы запускать AI на обычных видеокартах и смартфонах, применяют **квантование (Quantization)** — перевод вещественных чисел в форматы меньшей точности:

- **FP16 (Half precision)** — 16-битные вещественные числа. Ускоряют работу современных GPU в разы.  

- **BF16 (Brain Floating Point)** — специальный формат от Google. В нем сохранили размер экспоненты как у FP32, но урезали мантиссу. Это предотвращает выход чисел за границы (подходит для обучения LLM).  

- **INT8 / INT4** — перевод вещественных чисел в сверхкомпактные _целые_ числа. Модель теряет каплю точности, но начинает «летать» на мобильных процессорах.

FP32: [Знак: 1] [Экспонента: 8 бит] [Мантисса: 23 бита]  
FP16: [Знак: 1] [Экспонента: 5 бит] [Мантисса: 10 бит]  
BF16: [Знак: 1] [Экспонента: 8 бит] [Мантисса: 7 бит]

Из-за того, что у **FP16** под экспоненту выделено всего 5 бит, максимальное число, которое этот формат физически может закодировать, равно 65 504. Как только в процессе расчетов какое-то промежуточное число превышало 65 504, в формате FP16 оно мгновенно округлялось до Infinity (бесконечности).

>BF16 дал моделям способность не «взрываться» при огромных вычислениях. Как только компании (включая NVIDIA в видеокартах A100/H100) внедрили нативную поддержку BF16, индустрия смогла стабильно и без сбоев обучать модели масштаба GPT-4

#### Функции активации и специальные значения (NaN)

В AI постоянно используются сложные математические функции (например, _Sigmoid_, _Tanh_ или _Softmax_), где вещественные числа возводятся в степень (экспоненту $e^{x}$).

- **Проблема Infinity:** Если входное число слишком большое, функция $e^{x}$ мгновенно превратится в `Infinity` (выход за границы диапазона типа данных).  

- **Проблема NaN:** Как только в матрице весов появляется хотя бы одно `Infinity`, при операциях деления или вычитания (например, в функции потерь Loss) компьютер получает `NaN` (Not a Number). С этого момента вся нейросеть «ломается», так как любые дальнейшие математические операции с `NaN` тоже дают `NaN`. Ошибки «Loss: NaN» — один из главных кошмаров ML-инженеров при обучении моделей.

#### Компьютерное зрение (Computer Vision)

Цифровое изображение для ИИ — это не картинка, а матрица вещественных чисел.

- Изначально пиксели хранятся как целые числа от `0` до `255`.
- Перед подачей в нейросеть их **нормализуют** — переводят в вещественные числа в диапазоне от `0.0` до `1.0` (или от `-1.0` до `1.0`). Это нужно для того, чтобы алгоритмы оптимизации (например, Adam) работали стабильно и быстро.


#### Зачем вещественные числа возводят в степень экспоненту $e^{x}$ ?

Возведение вещественных чисел в степень экспоненты (число Эйлера e ~ 2.718) — это главный математический трюк в алгоритмах `классификации и финальных слоях нейросетей`. Чаще всего это происходит внутри функции **Softmax**.

Когда нейросеть (например, ChatGPT) пытается угадать следующее слово, на выходе ее скрытых слоев получаются сырые числовые значения — **логиты (logits)**. Эти логиты могут быть любыми вещественными числами: `-5.4`, `0.0`, `12.1`, `300.0`. Нам же нужно превратить этот хаос в **вероятности** (чтобы они были от 0 до 1, а их сумма равнялась 100%).

Для этого каждый логит возводят в степень экспоненты $e^{x}$ по трем фундаментальным причинам:

1. Избавление от отрицательных чисел

Вероятность не может быть отрицательной. Если мы просто поделим сырые значения на их сумму, отрицательные логиты дадут отрицательные вероятности, что абсурдно.  
Математическая функция \(y = e^x\) уникальна тем, что **ее результат всегда положителен**, какое бы число мы ни подставили вместо \(x\).

- $e^{5}$ ~ 148.4 (положительное)
- $e^{-5}$ ~ 0.0067 (тоже положительное, просто очень маленькое)

2. Экспоненциальное усиление контраста (Уверенность модели)

Экспонента работает как мощный увеличитель разницы между числами. Она помогает ИИ четко выбирать фаворита среди вариантов, а не сомневаться.  
Представьте два логита: `2.0` и `1.0`. Разница между ними всего в 1 балл.  
Если мы возведем их в степень \(e\):

- $e^{2.0}$ ~ 7.39
- $e^{1.0}$ ~ 2.72

Разница между результатами стала уже почти в 3 раза! Небольшое преимущество в сырых баллах экспонента превращает в безоговорочную победу в процентах вероятности, отсекая «неуверенность».

3. Идеальная производная для обучения

Нейросети учатся с помощью высшей математики (производных). Производная от функции \(e^{x}\) равна... самой функции \(e^{x}\)!  
Это свойство делает формулы обучения невероятно простыми, красивыми и вычислительно «дешевыми» для процессоров видеокарт. Компьютеру не нужно тратить такты на сложные тригонометрические или многочленные расчеты градиентов.

In [ ]:
# SOFTMAX
import numpy as np

def naive_softmax(x):
    """
    Опасная версия Softmax. 
    Сломается, если передать большие числа.
    """
    exps = np.exp(x)
    return exps / np.sum(exps)

def stable_softmax(x):
    """
    Безопасная версия Softmax (Промышленный стандарт).
    Защищена от переполнения памяти.
    """
    # 1. Находим максимум (работает как для одномерных массивов, так и для матриц)
    shift_x = x - np.max(x, axis=-1, keepdims=True)
    
    # 2. Возводим в степень экспоненты (теперь тут только отрицательные числа и 0)
    exps = np.exp(shift_x)
    
    # 3. Делим на сумму, чтобы получить вероятности
    return exps / np.sum(exps, axis=-1, keepdims=True)


# --- ТЕСТИРОВАНИЕ ---

# Обычные маленькие числа (обе функции работают хорошо)
small_logits = np.array([2.0, 1.0, 0.1])
print("Маленькие числа (Stable):", stable_softmax(small_logits))
# Выведет: [0.659, 0.242, 0.098] - это проценты вероятности для каждого варианта

# Гигантские числа (имитируем выброс или сбой при обучении нейросети)
huge_logits = np.array([1000.0, 999.0, 998.1])

try:
    print("Большие числа (Naive):", naive_softmax(huge_logits))
except Exception as e:
    print("Ошибка в naive_softmax:", e) 
    # В чистом Python тут будет OverflowError. 
    # В NumPy вы получите массив [nan, nan, nan] и предупреждение RuntimeWarning.

print("Большие числа (Stable):", stable_softmax(huge_logits))
# Выведет: [0.659, 0.242, 0.098] - магия! Результат точно такой же, и никакого NaN!

Для ИИ важна не абсолютная величина «сырых» баллов, а разница между ними. Вычитание максимума сжимает огромные вещественные числа в безопасный диапазон от -∞ до 0, предотвращая появление Infinity и последующее уничтожение весов модели через NaN.